# 🔬 StyleForge LLM — QLoRA Fine-Tuning Notebook

**Model**: Mistral-7B-Instruct-v0.3  
**Dataset**: `iamtarun/python_code_instructions_18k_alpaca` (Python coding)  
**Method**: QLoRA · 4-bit NF4 · r=16 · SFTTrainer  

## Prerequisites
- ✅ Runtime → Change runtime type → **A100 GPU** (recommended) or T4
- ✅ HuggingFace account + API token (for model download)
- ✅ ~40 GB disk space for model + checkpoints

## Estimated Time
| Hardware | 1 Epoch (5k samples) | 3 Epochs |
|----------|---------------------|----------|
| A100 40GB | ~25 min | ~75 min |
| T4 16GB   | ~70 min | ~210 min |
| V100 16GB | ~50 min | ~150 min |

## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q \
    torch>=2.1.0 \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    trl>=0.8.6 \
    bitsandbytes>=0.43.0 \
    datasets>=2.18.0 \
    huggingface_hub>=0.22.0 \
    rouge_score>=0.1.2 \
    accelerate>=0.28.0 \
    sentencepiece>=0.2.0

print('✅ Dependencies installed')

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2 — HuggingFace Login & Clone Repo

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Store your HF token in Colab Secrets (🔑 icon in left panel) as HF_TOKEN
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print('✅ HuggingFace login successful')
except Exception:
    print('⚠ Add HF_TOKEN to Colab Secrets, or paste it below:')
    HF_TOKEN = input('HuggingFace token: ').strip()
    login(token=HF_TOKEN)

# Clone your project (or upload files via the file panel)
import os
REPO_URL = 'YOUR_GITHUB_REPO_URL'  # <-- replace this
if REPO_URL != 'YOUR_GITHUB_REPO_URL':
    !git clone {REPO_URL} styleforge
    %cd styleforge/pipeline
else:
    print('📁 Upload pipeline/data_prep.py, train.py, eval.py manually or set REPO_URL')
    os.makedirs('/content/pipeline', exist_ok=True)
    %cd /content

## Cell 3 — Dataset Preparation

In [ ]:
# ── Configure ──────────────────────────────────────────────
HUB_USERNAME = 'your-hf-username'   # <-- set this
HUB_DATASET_REPO = f'{HUB_USERNAME}/python-coding-qlora-data'
MAX_SAMPLES = 5000

# ── Run data prep ──────────────────────────────────────────
!python /content/pipeline/data_prep.py \
    --dataset iamtarun/python_code_instructions_18k_alpaca \
    --max_samples {MAX_SAMPLES} \
    --max_tokens 1024 \
    --output_dir /content/data \
    --hf_token {HF_TOKEN}

# Optional: push to Hub
# !python /content/pipeline/data_prep.py \
#     --push_to_hub --hub_repo {HUB_DATASET_REPO} ...

# Verify output
!wc -l /content/data/train.jsonl /content/data/eval.jsonl
!head -1 /content/data/train.jsonl | python3 -c "import json,sys; d=json.load(sys.stdin); print(list(d.keys())); print(d['text'][:200])"

## Cell 4 — QLoRA Training

In [ ]:
# ── Configure ──────────────────────────────────────────────
EPOCHS     = 3         # reduce to 1 for a quick test run
BATCH_SIZE = 4         # reduce to 2 if OOM on T4
HUB_ADAPTER_REPO = f'{HUB_USERNAME}/mistral-7b-python-qlora'  # optional

# ── Train ──────────────────────────────────────────────────
!python /content/pipeline/train.py \
    --model_id mistralai/Mistral-7B-Instruct-v0.3 \
    --data_dir /content/data \
    --output_dir /content/checkpoints \
    --adapter_dir /content/adapter_weights \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --max_seq_length 1024 \
    --loss_log_path /content/training_loss.csv \
    --hf_token {HF_TOKEN}
    # --push_to_hub --hub_repo {HUB_ADAPTER_REPO}   # ← uncomment to push

print('\n✅ Training complete!')
!ls -lh /content/adapter_weights/

## Cell 5 — Visualise Training Loss

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/content/training_loss.csv')
print(df.tail(10))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df['step'], df['loss'], color='#8b5cf6', linewidth=1.5, alpha=0.9)
ax.fill_between(df['step'], df['loss'], alpha=0.12, color='#8b5cf6')
ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('Training Loss', fontsize=12)
ax.set_title('QLoRA Fine-Tuning Loss Curve', fontsize=14, fontweight='bold')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('/content/loss_curve.png', dpi=150)
plt.show()
print(f'Min loss: {df["loss"].min():.4f} @ step {df.loc[df["loss"].idxmin(), "step"]}')

## Cell 6 — Evaluation (ROUGE + Perplexity)

In [ ]:
!python /content/pipeline/eval.py \
    --model_id mistralai/Mistral-7B-Instruct-v0.3 \
    --data_dir /content/data \
    --adapter_dir /content/adapter_weights \
    --output_path /content/eval_results.json \
    --hf_token {HF_TOKEN}

import json
with open('/content/eval_results.json') as f:
    results = json.load(f)

print('\n=== Evaluation Results ===')
for r in results['results']:
    print(f"\n{r['model'].upper()}")
    print(f"  ROUGE-1: {r['rouge1']:.4f}  ROUGE-2: {r['rouge2']:.4f}  ROUGE-L: {r['rougeL']:.4f}")
    print(f"  Perplexity: {r['perplexity']}  Avg length: {r['avg_response_length']} words")

## Cell 7 — Download Artifacts

In [ ]:
# Zip everything you need for the inference server
!zip -r /content/styleforge_artifacts.zip \
    /content/adapter_weights \
    /content/training_loss.csv \
    /content/eval_results.json \
    /content/loss_curve.png

from google.colab import files
files.download('/content/styleforge_artifacts.zip')
print('\n✅ Artifacts downloaded!')
print('Next: extract zip → place adapter_weights/ in your server/ directory')
print('      place training_loss.csv + eval_results.json in project root')
print('      Run: cd server && uvicorn main:app --port 8000')